### Extracting a dataset, transforming it, and loading it into a CSV.

#### This section combines the imdb data with the principal people associated with the movies.

In [1]:
import pandas as pd

Getting the Kaggle imdb dataset.

In [2]:
imdb_df = pd.read_csv("movie-data/Imdb_Movie_Dataset.csv")

print(imdb_df.info())
print(imdb_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 21 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   id                    1048575 non-null  int64  
 1   title                 1048562 non-null  object 
 2   vote_average          1048575 non-null  float64
 3   vote_count            1048575 non-null  int64  
 4   status                1048575 non-null  object 
 5   release_date          866758 non-null   object 
 6   revenue               1048575 non-null  int64  
 7   runtime               1048575 non-null  int64  
 8   adult                 1048575 non-null  bool   
 9   budget                1048575 non-null  int64  
 10  imdb_id               560572 non-null   object 
 11  original_language     1048575 non-null  object 
 12  original_title        1048562 non-null  object 
 13  overview              832751 non-null   object 
 14  popularity            1048575 non-

Getting the dataset of principal people associated with the movie. (directors, actors, producers, etc.)

In [ ]:
principals_df = pd.read_csv("movie-data/imdb-pre-processed/title.principals.tsv", sep="\t", dtype=str)

print(principals_df.info())
print(principals_df.head(5))

Getting the dataset of people's names and personal information.

In [ ]:
names_df = pd.read_csv("movie-data/imdb-pre-processed/name.basics.tsv", sep="\t", dtype=str)

print(names_df.info())
print(names_df.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14208891 entries, 0 to 14208890
Data columns (total 6 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   nconst             object
 1   primaryName        object
 2   birthYear          object
 3   deathYear          object
 4   primaryProfession  object
 5   knownForTitles     object
dtypes: object(6)
memory usage: 650.4+ MB
None
      nconst      primaryName birthYear deathYear  \
0  nm0000001     Fred Astaire      1899      1987   
1  nm0000002    Lauren Bacall      1924      2014   
2  nm0000003  Brigitte Bardot      1934        \N   
3  nm0000004     John Belushi      1949      1982   
4  nm0000005   Ingmar Bergman      1918      2007   

                    primaryProfession                           knownForTitles  
0        actor,miscellaneous,producer  tt0072308,tt0050419,tt0027125,tt0031983  
1  actress,soundtrack,archive_footage  tt0037382,tt0075213,tt0117057,tt0038355  
2   actress,music_department,

 Gets rid of a few unecessary columns. (Otherwise the memory overload is too large)

In [ ]:
principals_df.drop(columns=["ordering", "job", "characters"], inplace=True)
names_df.drop(columns=["knownForTitles", "primaryProfession", "birthYear", "deathYear"], inplace=True)

print(principals_df.info())
print(names_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91093942 entries, 0 to 91093941
Data columns (total 3 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   tconst    object
 1   nconst    object
 2   category  object
dtypes: object(3)
memory usage: 2.0+ GB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14208891 entries, 0 to 14208890
Data columns (total 2 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   nconst       object
 1   primaryName  object
dtypes: object(2)
memory usage: 216.8+ MB
None


Combining the princpal people dataset with the name information dataset. (using the name id nconst as the index)

In [ ]:
full_names_df = pd.merge(principals_df, names_df, on="nconst", how="left")

print(full_names_df.info())
print(full_names_df.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91093942 entries, 0 to 91093941
Data columns (total 4 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   tconst       object
 1   nconst       object
 2   category     object
 3   primaryName  object
dtypes: object(4)
memory usage: 2.7+ GB
None
      tconst     nconst         category           primaryName
0  tt0000001  nm1588970             self            Carmencita
1  tt0000001  nm0005690         director  William K.L. Dickson
2  tt0000001  nm0005690         producer  William K.L. Dickson
3  tt0000001  nm0374658  cinematographer         William Heise
4  tt0000002  nm0721526         director         Émile Reynaud


This uses a left join on the title id of the movie, combining the full names data with the imdb dataset.

In [ ]:
combined_df = pd.merge(imdb_df, full_names_df, left_on="imdb_id", right_on="tconst", how="left")

print(combined_df.info())
print(combined_df.head(25).to_string())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8008881 entries, 0 to 8008880
Data columns (total 25 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    int64  
 1   title                 object 
 2   vote_average          float64
 3   vote_count            int64  
 4   status                object 
 5   release_date          object 
 6   revenue               int64  
 7   runtime               int64  
 8   adult                 bool   
 9   budget                int64  
 10  imdb_id               object 
 11  original_language     object 
 12  original_title        object 
 13  overview              object 
 14  popularity            float64
 15  tagline               object 
 16  genres                object 
 17  production_companies  object 
 18  production_countries  object 
 19  spoken_languages      object 
 20  keywords              object 
 21  tconst                object 
 22  nconst                object 
 23  categor

Creates a csv file to store the combined dataframe.

In [ ]:
combined_df.to_csv('movie-data/combined_df.csv', index=False)
del imdb_df, principals_df, names_df, full_names_df